# RL Peptide Optimizer — Results Report

This notebook analyses the results of the **DQN-based antimicrobial peptide optimiser** that navigates the [HydrAMP](https://github.com/szczurek-lab/HydrAMP) latent space to minimise E. coli MIC as predicted by [APEX](https://github.com/szczurek-lab/APEX).

---
## Model Description

### Problem statement
Given a seed antimicrobial peptide (AMP), find a mutant with **lower minimum inhibitory concentration (MIC)** against *Escherichia coli*.  
Lower MIC → more potent antibiotic activity.  The objective score is:

$$\text{score}(p) = \frac{1}{3}\sum_{i \in \{1,2,3\}} \log_2 \text{MIC}_i(p)$$

where indices 1, 2, 3 correspond to the three *E. coli* strains available in APEX (ATCC 11775, AIG221, AIG222).

### Architecture
| Component | Role |
|-----------|------|
| **HydrAMP encoder–decoder** | VAE mapping peptides ↔ 64-D latent vectors; used to encode states and candidate actions |
| **MUTANG++ (decoder Jacobian SVD)** | Local tangent-space mutation enumerator — proposes which residues can be substituted and with what probability |
| **DecoderLogProbPotential** | Scores each mutation by its log-probability under the HydrAMP decoder; used to rank and filter candidates |
| **APEX MIC predictor** | Black-box oracle returning predicted MIC (µM) for 11 pathogens; we use the log₂ mean over E. coli strains |
| **DQN agent** | Standard Deep Q-Network with experience replay and a hard-updated target network |

### State, action, reward
- **State** $s_t$: 64-D HydrAMP latent vector of the current peptide.
- **Action space** $\mathcal{A}(s_t)$: Up to `max_candidates` mutant peptides generated by MUTANG++, each represented by its own latent vector.  Only candidates whose softmax-normalised MUTANG++ probability exceeds $1/k$ (uniform baseline) are kept.
- **Reward** $r_t = \max(0,\; \text{ep\_best}_t - \text{score}(a_t))$: **Positive only when the chosen candidate sets a new episode-best score.**  The cumulative episode return therefore equals the total MIC improvement achieved in that episode: $\sum_t r_t = \text{score}(s_0) - \min_t \text{score}(s_t)$.
- **Horizon**: 20 steps per episode.

### Q-Network
```
input : [state (64) ‖ action (64)]  → dim 128
hidden: Linear(128,256) → ReLU → Linear(256,128) → ReLU → Linear(128,64) → ReLU
output: Linear(64,1)   → Q-value scalar
```
Action selection: $\epsilon$-greedy over Q-values evaluated on all candidates.

### Exploration schedule
$\epsilon$ is reset at the **start of each episode** using a linear schedule from `epsilon_start` (1.0) to `epsilon_end` (0.05) over the total number of episodes. This prevents the agent from collapsing into a single local mutation after the first few episodes.

---
## How to run

### Single peptide
```bash
cd pep-compass
python scripts/rl_peptide_optimizer.py run_rl_optimization \
    --start_peptide KTLKIIRLLF \
    --n_episodes 100 \
    --max_steps 20 \
    --max_candidates 40 \
    --device cuda \
    --output_dir results \
    --run_name mammuthusin-3
```

### All six benchmark peptides
```bash
python scripts/rl_peptide_optimizer.py run_all_peptides \
    --n_episodes 100 \
    --max_steps 20 \
    --device cuda \
    --output_dir results
```

### Smoke test
```bash
python scripts/rl_peptide_optimizer.py test_components --device cuda
```

### SLURM (one job per peptide, RTX 5000 Ada)
```bash
sbatch scripts/rl_slurm_array.sh
```

---
## Hyperparameters used

| Parameter | Value |
|-----------|-------|
| `n_episodes` | 100 |
| `max_steps` | 20 |
| `max_candidates` | 40 |
| `lr` | 1e-3 |
| `gamma` | 0.99 |
| `epsilon_start` | 1.0 |
| `epsilon_end` | 0.05 |
| `epsilon_decay` | 0.995 |
| `batch_size` | 32 |
| `buffer_capacity` | 10 000 |
| `target_update_freq` | 50 |
| Reward | trajectory-best improvement |
| Candidate filter | softmax prob > 1/k |


In [ ]:
# ── imports ────────────────────────────────────────────────────────────────
import json
import os
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

RESULTS_DIR = os.path.join(os.path.dirname(os.getcwd()), 'results')
# If running from the scripts/ folder:
if not os.path.isdir(RESULTS_DIR):
    RESULTS_DIR = 'results'
print(f'Results directory: {RESULTS_DIR}')

In [ ]:
# ── load all JSON result files ─────────────────────────────────────────────
json_files = sorted(glob.glob(os.path.join(RESULTS_DIR, '*_results.json')))
print(f'Found {len(json_files)} result file(s):')
for f in json_files:
    print(' ', os.path.basename(f))

runs = []
for fpath in json_files:
    with open(fpath) as fh:
        data = json.load(fh)
    data['_file'] = os.path.basename(fpath)
    runs.append(data)

print(f'\nLoaded {len(runs)} run(s).')

In [ ]:
# ── summary table ──────────────────────────────────────────────────────────
rows = []
for r in runs:
    start_mic = 2 ** r['start_score']
    best_mic  = 2 ** r['best_score']
    rows.append({
        'Run name':          r.get('run_name', r['_file']),
        'Start peptide':     r['start_peptide'],
        'Start score (log2)': round(r['start_score'], 4),
        'Start MIC (µM)':    round(start_mic, 1),
        'Best peptide':      r['best_peptide'],
        'Best score (log2)': round(r['best_score'], 4),
        'Best MIC (µM)':     round(best_mic, 1),
        'Improvement (log2)': round(r['improvement'], 4),
        'MIC fold-change':   round(start_mic / best_mic, 1),
        'Episodes':          len(r['episode_rewards']),
    })

df_summary = pd.DataFrame(rows)
display(df_summary.set_index('Run name'))

In [ ]:
# ── reward (= trajectory-best improvement) per episode ─────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharey=False)
axes = axes.flatten()

for ax, r in zip(axes, runs):
    name = r.get('run_name', r['_file'])
    ep_rewards = r['episode_rewards']
    best_scores = r['all_best_scores']
    
    # running global minimum
    global_min = np.minimum.accumulate(best_scores)
    
    episodes = np.arange(1, len(ep_rewards) + 1)
    ax2 = ax.twinx()
    ax.bar(episodes, ep_rewards, color='steelblue', alpha=0.45, label='Episode reward')
    ax2.plot(episodes, global_min, color='crimson', lw=2, label='Global best score')
    ax2.axhline(r['start_score'], color='grey', ls='--', lw=1, label='Start score')
    
    ax.set_title(f"{name}\n{r['start_peptide']} → {r['best_peptide']}", fontsize=9)
    ax.set_xlabel('Episode')
    ax.set_ylabel('Reward (log2-MIC drop)', color='steelblue')
    ax2.set_ylabel('Best score (log2 MIC)', color='crimson')
    ax.tick_params(axis='y', labelcolor='steelblue')
    ax2.tick_params(axis='y', labelcolor='crimson')

# hide unused axes
for ax in axes[len(runs):]:
    ax.set_visible(False)

fig.suptitle('DQN Peptide Optimizer — Episode Rewards & Global Best Score', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'reward_curves.png'), bbox_inches='tight')
plt.show()
print('Saved reward_curves.png')

In [ ]:
# ── per-peptide: episode-best score over episodes ─────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
colors = plt.cm.tab10.colors

for i, r in enumerate(runs):
    name = r.get('run_name', r['_file'])
    best_scores = r['all_best_scores']
    global_min = np.minimum.accumulate(best_scores)
    episodes = np.arange(1, len(best_scores) + 1)
    ax.plot(episodes, global_min, label=name, color=colors[i % 10], lw=1.8)
    ax.axhline(r['start_score'], color=colors[i % 10], ls=':', lw=0.9, alpha=0.6)

ax.set_xlabel('Episode')
ax.set_ylabel('Global best score (mean log2 MIC, E. coli)')
ax.set_title('Global Best Score Trajectory per Seed Peptide')
ax.legend(fontsize=9, ncol=2)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'global_best_trajectories.png'), bbox_inches='tight')
plt.show()
print('Saved global_best_trajectories.png')

In [ ]:
# ── MIC before / after bar chart ──────────────────────────────────────────
if runs:
    names = [r.get('run_name', r['_file']) for r in runs]
    start_mics = [2 ** r['start_score'] for r in runs]
    best_mics  = [2 ** r['best_score']  for r in runs]

    x = np.arange(len(names))
    w = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - w/2, start_mics, w, label='Start MIC (µM)', color='#aec6cf')
    ax.bar(x + w/2, best_mics,  w, label='Best MIC (µM)',  color='#77dd77')

    ax.set_yscale('log')
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=20, ha='right')
    ax.set_ylabel('Predicted MIC (µM, log scale)')
    ax.set_title('E. coli MIC Before and After DQN Optimisation')
    ax.legend()
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f'{y:.0f}'))
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'mic_comparison.png'), bbox_inches='tight')
    plt.show()
    print('Saved mic_comparison.png')

In [ ]:
# ── best peptides per run ──────────────────────────────────────────────────
for r in runs:
    name = r.get('run_name', r['_file'])
    top5_idx = np.argsort(r['all_best_scores'])[:5]
    print(f"\n{'='*55}")
    print(f"  {name}   start={r['start_peptide']}  score={r['start_score']:.4f}")
    print(f"  Top-5 episode-best peptides:")
    seen = set()
    for i in top5_idx:
        pep = r['all_best_peptides'][i]
        sc  = r['all_best_scores'][i]
        if pep not in seen:
            mic = 2**sc
            print(f"    ep{i+1:>3}  {pep:<22}  score={sc:.4f}  MIC={mic:.1f} µM")
            seen.add(pep)
print(f"\n{'='*55}")
print(f"  Global best across all runs:")
all_best = min(runs, key=lambda r: r['best_score'])
print(f"    {all_best['best_peptide']}  score={all_best['best_score']:.4f}  "
      f"MIC={2**all_best['best_score']:.1f} µM  (from {all_best.get('run_name','')}  start={all_best['start_peptide']})")

In [ ]:
# ── CSV log inspection ─────────────────────────────────────────────────────
csv_files = sorted(glob.glob(os.path.join(RESULTS_DIR, '*_log.csv')))
if csv_files:
    dfs = []
    for f in csv_files:
        df = pd.read_csv(f)
        df['_source'] = os.path.basename(f)
        dfs.append(df)
    df_all = pd.concat(dfs, ignore_index=True)

    print('Columns:', df_all.columns.tolist())
    print(f'Total rows: {len(df_all)}')
    display(df_all.groupby('start_peptide')[['ep_best_score','ep_reward']]
              .agg(['mean','min','max']).round(4))
else:
    print('No CSV log files found yet — run the optimiser first.')

In [ ]:
# ── (optional) run single peptide interactively ─────────────────────────
# Uncomment and run this cell to launch an optimisation directly from the notebook.

# import sys, os
# sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))
# from scripts.rl_peptide_optimizer import run_rl_optimization
#
# run_rl_optimization(
#     start_peptide='KTLKIIRLLF',
#     n_episodes=20,
#     max_steps=20,
#     device='cuda',
#     output_dir='results',
#     run_name='mammuthusin-3-test',
# )